In [7]:
# 01_ingestion.ipynb
# Purpose: Load raw IMD 2025 and MHCLG Revenue Outturn data into BigQuery
# Project: deprivation-spending-analysis
# Dataset: raw_data
# Author: Klea Zici, 19376977
# Date: July 2026
# Notes: IMD file has multiple sheets - using IMD sheet only
#        Revenue Outturn is CSV with status column - total rows will be filtered in staging
#        Both files loaded unmodified - all transformation happens in dbt

In [8]:
!pip install google-cloud-bigquery pandas openpyxl -q

In [9]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

project_id = 'deprivation-spending-analysis'
client = bigquery.Client(project=project_id)

print("Authentication successful")
print(f"Connected to project: {project_id}")

Authentication successful
Connected to project: deprivation-spending-analysis


In [10]:
import pandas as pd

# Load IMD 2025 File 10 - IMD sheet only
# Headers are on row 1, data starts row 2 - no rows to skip
imd_df = pd.read_excel(
    '/imd2025_file10_lad.xlsx',
    sheet_name='IMD'
)

print(f"IMD dataframe shape: {imd_df.shape}")
print(f"Columns: {list(imd_df.columns)}")
print(imd_df.head(3))

IMD dataframe shape: (296, 12)
Columns: ['Local Authority District code (2024)', 'Local Authority District name (2024)', 'IMD - Average rank ', 'IMD - Rank of average rank ', 'IMD - Average score ', 'IMD - Rank of average score ', 'IMD - Proportion of LSOAs in most deprived 10% nationally ', 'IMD - Rank of proportion of LSOAs in most deprived 10% nationally ', 'IMD25 - Extent ', 'IMD25 - Rank of extent ', 'IMD25 - Local concentration ', 'IMD25 - Rank of local concentration ']
  Local Authority District code (2024) Local Authority District name (2024)  \
0                            E06000001                           Hartlepool   
1                            E06000002                        Middlesbrough   
2                            E06000003                 Redcar and Cleveland   

   IMD - Average rank   IMD - Rank of average rank   IMD - Average score   \
0             23075.98                           30                37.580   
1             23840.54                          

In [11]:
import re

# Clean column names for BigQuery compatibility
def clean_column_name(col):
    col = col.lower()
    col = re.sub(r'[^a-z0-9_]', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    return col

imd_df.columns = [clean_column_name(c) for c in imd_df.columns]

print("Cleaned column names:")
print(list(imd_df.columns))

# Now load to BigQuery
table_id = 'deprivation-spending-analysis.raw_data.raw_imd_2025'
job = client.load_table_from_dataframe(imd_df, table_id)
job.result()

print(f"Loaded {job.output_rows} rows into {table_id}")

Cleaned column names:
['local_authority_district_code_2024', 'local_authority_district_name_2024', 'imd_average_rank', 'imd_rank_of_average_rank', 'imd_average_score', 'imd_rank_of_average_score', 'imd_proportion_of_lsoas_in_most_deprived_10_nationally', 'imd_rank_of_proportion_of_lsoas_in_most_deprived_10_nationally', 'imd25_extent', 'imd25_rank_of_extent', 'imd25_local_concentration', 'imd25_rank_of_local_concentration']
Loaded 296 rows into deprivation-spending-analysis.raw_data.raw_imd_2025


In [12]:
# ============================================================
# LOAD REVENUE OUTTURN CSV
# ============================================================
# This file contains local authority spending data for England
# covering multiple financial years in a single flat CSV file.
# Each row represents one local authority in one year.
# The 'status' column contains 'submitted' (individual authority data)
# and 'total' (aggregate rows we must filter out later in dbt staging).
# Year format is YYYYMM — so 202003 = financial year ending March 2020 = 2019-20.
# We load it raw here without filtering — all cleaning happens in dbt.
# ============================================================

revenue_df = pd.read_csv(
    '/mhclg_revenue_outturn_multiyear.csv.csv',
    encoding='utf-8',
    low_memory=False  # prevents mixed-type warnings on large files
)

# Inspect the shape and key columns before loading to BigQuery
print(f"Revenue Outturn shape: {revenue_df.shape}")
print(f"First 10 columns: {list(revenue_df.columns[:10])}")
print(f"Status values (should be submitted and total): {revenue_df['status'].unique()}")
print(f"Year values present: {revenue_df['year_ending'].unique()}")
print(revenue_df.head(3))

Revenue Outturn shape: (3504, 2323)
First 10 columns: ['year_ending', 'ONS_code', 'LA_LGF_code', 'LA_name', 'status', 'LA_class', 'LA_subclass', 'RG_grantindsg_tot_grant', 'RG_grantinppg_tot_grant', 'RG_grantinusm_tot_grant']
Status values (should be submitted and total): ['total' 'submitted' 'not submitted']
Year values present: [201803 201903 202003 202103 202203 202303 202403 202503]
   year_ending   ONS_code LA_LGF_code              LA_name     status  \
0       201803        E06          UA  Unitary Authorities      total   
1       201803  E06000001       E0701        Hartlepool UA  submitted   
2       201803  E06000002       E0702     Middlesbrough UA  submitted   

            LA_class        LA_subclass  RG_grantindsg_tot_grant  \
0  Unitary Authority  Unitary Authority                5436305.0   
1  Unitary Authority  Unitary Authority                  47938.0   
2  Unitary Authority  Unitary Authority                  63793.0   

   RG_grantinppg_tot_grant  RG_grantinusm_to

In [13]:
# ============================================================
# INVESTIGATE 'NOT SUBMITTED' ROWS
# ============================================================
# We discovered three status values: total, submitted, not submitted.
# Before loading to BigQuery we need to understand which authorities
# did not submit and in which years — this affects the analysis directly.
# An authority that did not submit will be absent from the joined dataset.
# This is a discretionary decision point documented in catalogue entry D11.
# ============================================================

# Count rows by status
print("Row counts by status:")
print(revenue_df['status'].value_counts())

print("\nNot submitted authorities by year:")
not_submitted = revenue_df[revenue_df['status'] == 'not submitted']
print(not_submitted[['year_ending', 'ONS_code', 'LA_name']].to_string())

Row counts by status:
status
submitted        3407
total              74
not submitted      23
Name: count, dtype: int64

Not submitted authorities by year:
      year_ending   ONS_code                           LA_name
2719       202403  E06000064           Westmorland and Furness
2723       202403  E07000008                         Cambridge
2751       202403  E07000068                         Brentwood
2752       202403  E07000069                      Castle Point
2754       202403  E07000071                        Colchester
2801       202403  E07000122                            Pendle
2813       202403  E07000134         North West Leicestershire
2841       202403  E07000192                     Cannock Chase
2846       202403  E07000197                       Stafford BC
2849       202403  E07000200                           Babergh
2851       202403  E07000203                       Mid Suffolk
2862       202403  E07000217                         Woking BC
2911       202403  E0800

In [14]:
# ============================================================
# CLEAN COLUMN NAMES AND LOAD REVENUE OUTTURN TO BIGQUERY
# ============================================================
# Loading the full raw file unfiltered — status filtering happens in dbt staging.
# Column names are already machine-readable unlike the IMD file,
# but we apply the same cleaning function for consistency.
# ONS_code will be the join key with the IMD table.
# 2323 columns — vast majority are spending category codes.
# ============================================================

# Apply same column cleaning function as IMD for consistency
revenue_df.columns = [clean_column_name(c) for c in revenue_df.columns]

print("First 10 cleaned column names:")
print(list(revenue_df.columns[:10]))

# Load full raw file to BigQuery — no filtering at this stage
table_id = 'deprivation-spending-analysis.raw_data.raw_revenue_outturn'

job = client.load_table_from_dataframe(
    revenue_df,
    table_id,
    job_config=bigquery.LoadJobConfig(
        write_disposition='WRITE_TRUNCATE'  # overwrite if table exists
    )
)
job.result()

print(f"Loaded {job.output_rows} rows into {table_id}")

First 10 cleaned column names:
['year_ending', 'ons_code', 'la_lgf_code', 'la_name', 'status', 'la_class', 'la_subclass', 'rg_grantindsg_tot_grant', 'rg_grantinppg_tot_grant', 'rg_grantinusm_tot_grant']
Loaded 3504 rows into deprivation-spending-analysis.raw_data.raw_revenue_outturn


In [15]:
# ============================================================
# VERIFY BOTH TABLES EXIST IN BIGQUERY
# ============================================================
# Quick row count check to confirm both tables loaded correctly
# before moving to dbt staging phase.
# ============================================================

tables = ['raw_imd_2025', 'raw_revenue_outturn']

for table in tables:
    query = f"SELECT COUNT(*) as row_count FROM `deprivation-spending-analysis.raw_data.{table}`"
    result = client.query(query).result()
    for row in result:
        print(f"{table}: {row.row_count} rows confirmed in BigQuery")

raw_imd_2025: 592 rows confirmed in BigQuery
raw_revenue_outturn: 3504 rows confirmed in BigQuery


In [16]:
pop_df = pd.read_excel(
    '/mye24tablesuk.xlsx',
    sheet_name='MYE4',
    header=7
)

print(f"Shape: {pop_df.shape}")
print(f"Columns: {list(pop_df.columns[:6])}")
print(pop_df.head(3))

Shape: (404, 17)
Columns: ['Code', 'Name', 'Geography', 'Mid-2024', 'Mid-2023', 'Mid-2022']
        Code               Name Geography  Mid-2024  Mid-2023  Mid-2022  \
0  K02000001     UNITED KINGDOM   Country  69281437  68526183  67636134   
1  K03000001      GREAT BRITAIN   Country  67353582  66605801  65725591   
2  K04000001  ENGLAND AND WALES   Country  61806682  61099801  60278591   

   Mid-2021  Mid-2020  Mid-2019  Mid-2018  Mid-2017  Mid-2016  Mid-2015  \
0  66977988  66739867  66627507  66286727  65964292  65605841  65086958   
1  65073424  64839344  64728988  64400468  64089114  63739799  63232015   
2  59660524  59430444  59317788  59008368  58700914  58366199  57881415   

   Mid-2014  Mid-2013  Mid-2012  Mid-2011  
0  64618693  64138221  63710543  63285145  
1  62775507  62306544  61885940  61470827  
2  57444107  56989744  56577740  56170927  


In [17]:
# ============================================================
# FILTER AND RESHAPE POPULATION DATA
# ============================================================
# Keep only English local authority districts (codes starting with E)
# Exclude country, region, and county level rows
# Keep only years needed for study period 2019-20 to 2024-25
# Reshape from wide to long format for joining with spending data
# ============================================================

# Filter to England local authority districts only
# English LAD codes start with E06, E07, E08, E09
# Exclude E92 (England country), E12 (regions), E10/E11 (counties/met counties)
england_lad_codes = ['E06', 'E07', 'E08', 'E09']
pop_england = pop_df[
    pop_df['Code'].str[:3].isin(england_lad_codes)
].copy()

print(f"England LAD rows: {len(pop_england)}")
print(pop_england.head(3))

# Keep only relevant columns
years_needed = ['Code', 'Name', 'Mid-2019', 'Mid-2020', 'Mid-2021',
                'Mid-2022', 'Mid-2023', 'Mid-2024']
pop_filtered = pop_england[years_needed].copy()

# Reshape from wide to long format
pop_long = pop_filtered.melt(
    id_vars=['Code', 'Name'],
    var_name='mid_year',
    value_name='population'
)

# Create financial_year_start column
# Mid-2019 population used for 2019-20, Mid-2020 for 2020-21, etc
pop_long['financial_year_start'] = pop_long['mid_year'].str.replace('Mid-', '').astype(int)

# Rename columns to match staging model
pop_long = pop_long.rename(columns={'Code': 'lad_code', 'Name': 'lad_name'})
pop_long = pop_long[['lad_code', 'lad_name', 'financial_year_start', 'population']]

print(f"\nReshaped population data shape: {pop_long.shape}")
print(pop_long.head(6))

England LAD rows: 296
        Code           Name          Geography  Mid-2024  Mid-2023  Mid-2022  \
5  E06000047  County Durham  Unitary Authority    538011    533997    527755   
6  E06000005     Darlington  Unitary Authority    112489    110925    109461   
7  E06000001     Hartlepool  Unitary Authority     98180     95942     94161   

   Mid-2021  Mid-2020  Mid-2019  Mid-2018  Mid-2017  Mid-2016  Mid-2015  \
5    521447    519204    518562    516648    515911    514431    513924   
6    108210    106827    106532    106411    106541    106258    105988   
7     92575     92202     92401     92288     92306     92260     92125   

   Mid-2014  Mid-2013  Mid-2012  Mid-2011  
5    513604    513179    513122    512994  
6    105825    105591    105557    105584  
7     92358     92465     92344     92088  

Reshaped population data shape: (1776, 4)
    lad_code              lad_name  financial_year_start  population
0  E06000047         County Durham                  2019      518562

In [18]:
# ============================================================
# LOAD POPULATION DATA TO BIGQUERY
# ============================================================
# Loading reshaped long-format population data
# 296 English local authority districts x 6 years = 1776 rows
# lad_code joins to IMD and Revenue Outturn tables
# financial_year_start joins to year in Revenue Outturn
# ============================================================

# Clean column names for BigQuery compatibility
pop_long.columns = [clean_column_name(c) for c in pop_long.columns]

print("Column names:", list(pop_long.columns))

# Load to BigQuery
table_id = 'deprivation-spending-analysis.raw_data.raw_population'

job = client.load_table_from_dataframe(
    pop_long,
    table_id,
    job_config=bigquery.LoadJobConfig(
        write_disposition='WRITE_TRUNCATE'
    )
)
job.result()

print(f"Loaded {job.output_rows} rows into {table_id}")

Column names: ['lad_code', 'lad_name', 'financial_year_start', 'population']
Loaded 1776 rows into deprivation-spending-analysis.raw_data.raw_population


In [19]:
# ============================================================
# VERIFY RS_ AGGREGATE COLUMNS EXIST IN REVENUE OUTTURN DATA
# ============================================================
# Checking for the three key spending variables identified:
# RS_totsx_net_exp - Total Service Expenditure
# RS_asc_net_exp - Adult Social Care
# RS_csc_net_exp - Children's Social Care
# These are MHCLG's documented aggregate measures
# ============================================================

import pandas as pd

rev_df = pd.read_csv('/mhclg_revenue_outturn_multiyear.csv.csv',
                     low_memory=False)

# Search for RS_ columns
rs_cols = [col for col in rev_df.columns if col.startswith('RS_') or col.lower().startswith('rs_')]
print(f"Total RS_ columns found: {len(rs_cols)}")
print("All RS_ columns:")
print(rs_cols)

# Check specifically for our target columns
target_cols = ['RS_totsx_net_exp', 'RS_asc_net_exp', 'RS_csc_net_exp']
for col in target_cols:
    if col in rev_df.columns:
        print(f"✓ {col} EXISTS")
    else:
        print(f"✗ {col} NOT FOUND")

Total RS_ columns found: 189
All RS_ columns:
['RS_edu_net_exp', 'RS_trans_net_exp', 'RS_csc_net_exp', 'RS_asc_net_exp', 'RS_phs_net_exp', 'RS_hous_net_exp', 'RS_cul_net_exp', 'RS_env_net_exp', 'RS_plan_net_exp', 'RS_pol_net_exp', 'RS_frs_net_exp', 'RS_cen_net_exp', 'RS_oth_net_exp', 'RS_totsx_net_exp', 'RS_hbrent_net_exp', 'RS_hbnreb_net_exp', 'RS_hbhreb_net_exp', 'RS_hbsubl_net_exp', 'RS_hbhrashare_net_exp', 'RS_parishaggprecept_net_exp', 'RS_levyita_net_exp', 'RS_levywaste_net_exp', 'RS_levylonpen_net_exp', 'RS_levyother_net_exp', 'RS_tradaccext_net_exp', 'RS_tradaccint_net_exp', 'RS_tradcapext_net_exp', 'RS_tradcapint_net_exp', 'RS_accumul_net_exp', 'RS_adjnce_net_exp', 'RS_netcurrtot_net_exp', 'RS_levyflood_net_exp', 'RS_cera_net_exp', 'RS_ceraph_net_exp', 'RS_caprecflex_net_exp', 'RS_adjexpcap_net_exp', 'RS_provbad_net_exp', 'RS_provprin_net_exp', 'RS_leasing_net_exp', 'RS_intextpay_net_exp', 'RS_inthrapay_net_exp', 'RS_sub_tot_net_exp', 'RS_intinvinc_net_exp', 'RS_pfidiff_net_ex

In [20]:
# ============================================================
# COMPARE SPENDING AGGREGATE VARIABLES FOR HARTLEPOOL
# ============================================================
# Checking RS_totsx_net_exp vs RS_netcurrtot_net_exp
# to confirm which is closest to academic literature measure
# Hartlepool used as reference authority (first in IMD file)
# ============================================================

hartlepool = rev_df[
    (rev_df['ONS_code'] == 'E06000001') &
    (rev_df['status'] == 'submitted') &
    (rev_df['year_ending'] == 202003)
][['ONS_code', 'LA_name', 'year_ending',
   'RS_totsx_net_exp', 'RS_asc_net_exp',
   'RS_csc_net_exp', 'RS_netcurrtot_net_exp',
   'RS_netrevexp_net_exp']]

print(hartlepool.to_string())

      ONS_code        LA_name  year_ending  RS_totsx_net_exp  RS_asc_net_exp  RS_csc_net_exp  RS_netcurrtot_net_exp  RS_netrevexp_net_exp
904  E06000001  Hartlepool UA       202003          160314.0         32680.0         24796.0               187847.0               84676.0


In [21]:
# ============================================================
# CHECK WHETHER GAP BETWEEN LINE 749 AND LINE 699
# CORRELATES WITH DEPRIVATION
# ============================================================

import pandas as pd
import numpy as np

rev_df = pd.read_csv('/mhclg_revenue_outturn_multiyear.csv.csv',
                     low_memory=False)

imd_df = pd.read_excel('/imd2025_file10_lad.xlsx',
                       sheet_name='IMD')

# Filter revenue to 2019-20 submitted rows only
rev_2019 = rev_df[
    (rev_df['year_ending'] == 202003) &
    (rev_df['status'] == 'submitted')
][['ONS_code', 'RS_netcurrtot_net_exp', 'RS_totsx_net_exp']].copy()

# Calculate gap
rev_2019['gap'] = rev_2019['RS_netcurrtot_net_exp'] - rev_2019['RS_totsx_net_exp']
rev_2019['gap_pct'] = (rev_2019['gap'] / rev_2019['RS_totsx_net_exp']) * 100

# Clean IMD column names
imd_df.columns = [c.lower().replace(' ', '_').replace('(', '').replace(')', '') for c in imd_df.columns]

# Use correct column name
imd_clean = imd_df[['local_authority_district_code_2024', 'imd_-_average_score_']].copy()
imd_clean.columns = ['ONS_code', 'imd_score']

# Merge
merged = rev_2019.merge(imd_clean, on='ONS_code', how='inner')

# Correlation
corr = merged['gap_pct'].corr(merged['imd_score'])
print(f"Correlation between gap% and IMD score: {corr:.3f}")
print(f"Number of authorities: {len(merged)}")
print(f"Mean gap%: {merged['gap_pct'].mean():.1f}%")
print(f"Std gap%: {merged['gap_pct'].std():.1f}%")
print(merged[['ONS_code', 'imd_score', 'RS_netcurrtot_net_exp',
              'RS_totsx_net_exp', 'gap_pct']].head(5).to_string())

Correlation between gap% and IMD score: -0.292
Number of authorities: 289
Mean gap%: 102.4%
Std gap%: 80.5%
    ONS_code  imd_score  RS_netcurrtot_net_exp  RS_totsx_net_exp    gap_pct
0  E06000001     37.580               187847.0          160314.0  17.174420
1  E06000002     40.037               272697.0          215818.0  26.355077
2  E06000003     30.336               234533.0          188528.0  24.402211
3  E06000004     25.830               311376.0          250434.0  24.334555
4  E06000005     25.713               157401.0          129672.0  21.383953


In [22]:
# ============================================================
# VERIFY MART TABLE
# ============================================================
from google.cloud import bigquery
client = bigquery.Client(project='deprivation-spending-analysis')

query = """
SELECT
    financial_year_start,
    COUNT(DISTINCT lad_code) as authority_count,
    COUNT(*) as row_count,
    ROUND(AVG(imd_score), 2) as avg_imd_score,
    ROUND(AVG(net_current_expenditure_per_capita), 2) as avg_spending_per_capita
FROM `deprivation-spending-analysis.spending_analysis_marts.mart_deprivation_spending`
GROUP BY financial_year_start
ORDER BY financial_year_start
"""

result = client.query(query).result()
for row in result:
    print(f"{row.financial_year_start}: {row.authority_count} authorities, "
          f"avg IMD={row.avg_imd_score}, "
          f"avg spending per capita=£{row.avg_spending_per_capita}")

2019: 289 authorities, avg IMD=20.14, avg spending per capita=£1044.2
2020: 290 authorities, avg IMD=20.11, avg spending per capita=£1126.54
2021: 292 authorities, avg IMD=20.09, avg spending per capita=£1107.48
2022: 292 authorities, avg IMD=20.09, avg spending per capita=£1122.3
2023: 283 authorities, avg IMD=20.16, avg spending per capita=£1232.03
2024: 287 authorities, avg IMD=20.07, avg spending per capita=£1270.44


In [23]:
# ============================================================
# INVESTIGATE WHICH AUTHORITIES ARE EXCLUDED FROM MART TABLE
# ============================================================
# The inner join excludes authorities not present in both
# IMD and Revenue Outturn. Need to document which ones and why.
# ============================================================

from google.cloud import bigquery
client = bigquery.Client(project='deprivation-spending-analysis')

# Authorities in Revenue Outturn submitted rows for study period
# but NOT in the mart table
query = """
WITH revenue_lads AS (
    SELECT DISTINCT ons_code as lad_code, year_ending
    FROM `deprivation-spending-analysis.raw_data.raw_revenue_outturn`
    WHERE status = 'submitted'
    AND year_ending BETWEEN 202003 AND 202503
),
mart_lads AS (
    SELECT DISTINCT lad_code,
    CAST(financial_year_start AS STRING) || '03' as year_str
    FROM `deprivation-spending-analysis.spending_analysis_marts.mart_deprivation_spending`
),
imd_lads AS (
    SELECT DISTINCT local_authority_district_code_2024 as lad_code
    FROM `deprivation-spending-analysis.raw_data.raw_imd_2025`
)

SELECT
    r.lad_code,
    r.year_ending,
    CASE WHEN i.lad_code IS NULL THEN 'Not in IMD' ELSE 'In IMD' END as imd_status
FROM revenue_lads r
LEFT JOIN imd_lads i ON r.lad_code = i.lad_code
WHERE i.lad_code IS NULL
ORDER BY r.lad_code, r.year_ending
"""

result = client.query(query).result()
rows = list(result)
print(f"Authorities in Revenue Outturn but NOT in IMD: {len(rows)}")
for row in rows[:20]:
    print(f"{row.lad_code} — year {row.year_ending} — {row.imd_status}")

Authorities in Revenue Outturn but NOT in IMD: 785
E07000004 — year 202003 — Not in IMD
E07000005 — year 202003 — Not in IMD
E07000006 — year 202003 — Not in IMD
E07000007 — year 202003 — Not in IMD
E07000026 — year 202003 — Not in IMD
E07000026 — year 202103 — Not in IMD
E07000026 — year 202203 — Not in IMD
E07000026 — year 202303 — Not in IMD
E07000027 — year 202003 — Not in IMD
E07000027 — year 202103 — Not in IMD
E07000027 — year 202203 — Not in IMD
E07000027 — year 202303 — Not in IMD
E07000028 — year 202003 — Not in IMD
E07000028 — year 202103 — Not in IMD
E07000028 — year 202203 — Not in IMD
E07000028 — year 202303 — Not in IMD
E07000029 — year 202003 — Not in IMD
E07000029 — year 202103 — Not in IMD
E07000029 — year 202203 — Not in IMD
E07000029 — year 202303 — Not in IMD


In [24]:
# ============================================================
# INVESTIGATE EXCLUDED AUTHORITIES BY TYPE
# ============================================================
# 785 authorities in Revenue Outturn not in IMD
# Need to understand what types they are
# ============================================================

query = """
WITH revenue_lads AS (
    SELECT DISTINCT ons_code as lad_code, la_name, la_class, la_subclass
    FROM `deprivation-spending-analysis.raw_data.raw_revenue_outturn`
    WHERE status = 'submitted'
    AND year_ending = 202003
),
imd_lads AS (
    SELECT DISTINCT local_authority_district_code_2024 as lad_code
    FROM `deprivation-spending-analysis.raw_data.raw_imd_2025`
)

SELECT
    r.la_class,
    r.la_subclass,
    COUNT(*) as count
FROM revenue_lads r
LEFT JOIN imd_lads i ON r.lad_code = i.lad_code
WHERE i.lad_code IS NULL
GROUP BY r.la_class, r.la_subclass
ORDER BY count DESC
"""

result = client.query(query).result()
print("Authority types excluded by inner join:")
for row in result:
    print(f"{row.la_class} / {row.la_subclass}: {row.count}")

Authority types excluded by inner join:
Other / Police: 37
Other / Fire: 29
Shire District / Shire District: 28
Shire County / Shire County: 26
Other / Park: 11
Other / Combined Authority: 10
Other / Waste: 5
Other / GLA: 1


In [25]:
# ============================================================
# INVESTIGATE SHIRE DISTRICT CODE MISMATCHES
# ============================================================
# Shire districts should be in IMD but are not matching
# Need to see their actual ONS codes to diagnose the problem
# ============================================================

query = """
WITH revenue_lads AS (
    SELECT DISTINCT ons_code as lad_code, la_name, la_class
    FROM `deprivation-spending-analysis.raw_data.raw_revenue_outturn`
    WHERE status = 'submitted'
    AND year_ending = 202003
    AND la_class = 'Shire District'
),
imd_lads AS (
    SELECT DISTINCT local_authority_district_code_2024 as lad_code
    FROM `deprivation-spending-analysis.raw_data.raw_imd_2025`
)

SELECT
    r.lad_code,
    r.la_name,
    CASE WHEN i.lad_code IS NULL THEN 'NOT IN IMD' ELSE 'IN IMD' END as status
FROM revenue_lads r
LEFT JOIN imd_lads i ON r.lad_code = i.lad_code
WHERE i.lad_code IS NULL
ORDER BY r.lad_code
LIMIT 20
"""

result = client.query(query).result()
print("Shire Districts in Revenue Outturn NOT matching IMD:")
for row in result:
    print(f"{row.lad_code} — {row.la_name}")

Shire Districts in Revenue Outturn NOT matching IMD:
E07000004 — Aylesbury Vale DC
E07000005 — Chiltern
E07000006 — South Buckinghamshire
E07000007 — Wycombe
E07000026 — Allerdale
E07000027 — Barrow-in-Furness
E07000028 — Carlisle
E07000029 — Copeland
E07000030 — Eden
E07000031 — South Lakeland
E07000150 — Corby
E07000151 — Daventry DC
E07000152 — East Northamptonshire
E07000153 — Kettering
E07000154 — Northampton
E07000155 — South Northamptonshire
E07000156 — Wellingborough
E07000163 — Craven
E07000164 — Hambleton
E07000165 — Harrogate


In [26]:
# ============================================================
# CHECK FOR DUPLICATE LAD CODES IN IMD STAGING VIEW
# ============================================================

query = """
SELECT
    lad_code,
    COUNT(*) as count
FROM `deprivation-spending-analysis.spending_analysis_staging.stg_imd_2025`
GROUP BY lad_code
HAVING COUNT(*) > 1
ORDER BY count DESC
LIMIT 10
"""

result = client.query(query).result()
rows = list(result)
print(f"LAD codes appearing more than once: {len(rows)}")
if rows:
    for row in rows[:5]:
        print(f"{row.lad_code}: {row.count} times")
else:
    print("No duplicates found")

LAD codes appearing more than once: 10
E08000030: 2 times
E07000076: 2 times
E07000062: 2 times
E06000002: 2 times
E06000021: 2 times


In [27]:
# ============================================================
# CHECK DUPLICATES IN RAW IMD TABLE
# ============================================================

query = """
SELECT
    local_authority_district_code_2024,
    local_authority_district_name_2024,
    COUNT(*) as count
FROM `deprivation-spending-analysis.raw_data.raw_imd_2025`
GROUP BY
    local_authority_district_code_2024,
    local_authority_district_name_2024
HAVING COUNT(*) > 1
ORDER BY count DESC
"""

result = client.query(query).result()
rows = list(result)
print(f"Duplicate codes in raw IMD table: {len(rows)}")
for row in rows:
    print(f"{row.local_authority_district_code_2024} — {row.local_authority_district_name_2024}: {row.count} times")

Duplicate codes in raw IMD table: 296
E07000238 — Wychavon: 2 times
E08000003 — Manchester: 2 times
E06000009 — Blackpool: 2 times
E08000004 — Oldham: 2 times
E08000012 — Liverpool: 2 times
E07000120 — Hyndburn: 2 times
E06000006 — Halton: 2 times
E08000024 — Sunderland: 2 times
E08000013 — St. Helens: 2 times
E08000019 — Sheffield: 2 times
E08000031 — Wolverhampton: 2 times
E08000016 — Barnsley: 2 times
E08000023 — South Tyneside: 2 times
E06000016 — Leicester: 2 times
E06000004 — Stockton-on-Tees: 2 times
E08000014 — Sefton: 2 times
E08000008 — Tameside: 2 times
E07000061 — Eastbourne: 2 times
E06000044 — Portsmouth: 2 times
E07000148 — Norwich: 2 times
E07000199 — Tamworth: 2 times
E07000108 — Dover: 2 times
E07000121 — Lancaster: 2 times
E07000142 — West Lindsey: 2 times
E06000033 — Southend-on-Sea: 2 times
E06000027 — Torbay: 2 times
E06000035 — Medway: 2 times
E06000050 — Cheshire West and Chester: 2 times
E08000022 — North Tyneside: 2 times
E07000066 — Basildon: 2 times
E0800000

In [28]:
# ============================================================
# RELOAD IMD TABLE WITH WRITE_TRUNCATE TO REMOVE DUPLICATES
# ============================================================
# Table was loaded multiple times causing duplicate rows
# WRITE_TRUNCATE overwrites the table completely with clean data
# ============================================================

import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project='deprivation-spending-analysis')

# Reload IMD file
imd_df = pd.read_excel(
    '/imd2025_file10_lad.xlsx',
    sheet_name='IMD'
)

# Clean column names
import re
def clean_column_name(col):
    col = str(col).lower()
    col = re.sub(r'[^a-z0-9_]', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    return col

imd_df.columns = [clean_column_name(c) for c in imd_df.columns]

print(f"IMD rows to load: {len(imd_df)}")

# Load with WRITE_TRUNCATE to overwrite existing table
table_id = 'deprivation-spending-analysis.raw_data.raw_imd_2025'

job = client.load_table_from_dataframe(
    imd_df,
    table_id,
    job_config=bigquery.LoadJobConfig(
        write_disposition='WRITE_TRUNCATE'
    )
)
job.result()

print(f"Reloaded {job.output_rows} rows into {table_id}")
print("Duplicates removed")

IMD rows to load: 296
Reloaded 296 rows into deprivation-spending-analysis.raw_data.raw_imd_2025
Duplicates removed


In [29]:
# ============================================================
# VERIFY MART TABLE AFTER IMD RELOAD
# ============================================================

query = """
SELECT
    financial_year_start,
    COUNT(DISTINCT lad_code) as authority_count,
    COUNT(*) as row_count
FROM `deprivation-spending-analysis.spending_analysis_marts.mart_deprivation_spending`
GROUP BY financial_year_start
ORDER BY financial_year_start
"""

result = client.query(query).result()
for row in result:
    print(f"{row.financial_year_start}: {row.authority_count} authorities, {row.row_count} rows")

2019: 289 authorities, 289 rows
2020: 290 authorities, 290 rows
2021: 292 authorities, 292 rows
2022: 292 authorities, 292 rows
2023: 283 authorities, 283 rows
2024: 287 authorities, 287 rows


In [30]:
# ============================================================
# LOAD ONS REGION LOOKUP INTO BIGQUERY
# ============================================================
# Source: ONS Local Authority District to Region December 2024
# Contains LAD code to region name mapping for England
# Used as regional control variable in regression (Decision D18)
# Published under OGL v3.0
# ============================================================

import pandas as pd
from google.cloud import bigquery
import re

client = bigquery.Client(project='deprivation-spending-analysis')

def clean_column_name(col):
    col = str(col).lower()
    col = re.sub(r'[^a-z0-9_]', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    return col

# Load region lookup
region_df = pd.read_csv('/Local_Authority_District_to_Region_(December_2024)_Lookup_in_EN (2).csv')

print(f"Shape: {region_df.shape}")
print(f"Columns: {list(region_df.columns)}")
print(region_df.head(3))

Shape: (296, 5)
Columns: ['LAD24CD', 'LAD24NM', 'RGN24CD', 'RGN24NM', 'ObjectId']
     LAD24CD               LAD24NM    RGN24CD     RGN24NM  ObjectId
0  E06000001            Hartlepool  E12000001  North East         1
1  E06000002         Middlesbrough  E12000001  North East         2
2  E06000003  Redcar and Cleveland  E12000001  North East         3


In [31]:
# ============================================================
# LOAD ONS REGION LOOKUP INTO BIGQUERY
# ============================================================
# Source: ONS LAD to Region December 2024 lookup
# 296 English local authority districts with region names
# Used as regional control variable in regression (Decision D18)
# Published under OGL v3.0
# ============================================================

import pandas as pd
from google.cloud import bigquery
import re

client = bigquery.Client(project='deprivation-spending-analysis')

def clean_column_name(col):
    col = str(col).lower()
    col = re.sub(r'[^a-z0-9_]', '_', col)
    col = re.sub(r'_+', '_', col)
    col = col.strip('_')
    return col

# Keep only LAD code and region name
region_df = pd.read_csv('/Local_Authority_District_to_Region_(December_2024)_Lookup_in_EN (2).csv')
region_df = region_df[['LAD24CD', 'RGN24NM']].copy()
region_df.columns = ['lad_code', 'region_name']

print(f"Shape: {region_df.shape}")
print(f"Unique regions: {sorted(region_df['region_name'].unique())}")

# Load to BigQuery
table_id = 'deprivation-spending-analysis.raw_data.raw_region_lookup'

job = client.load_table_from_dataframe(
    region_df,
    table_id,
    job_config=bigquery.LoadJobConfig(
        write_disposition='WRITE_TRUNCATE'
    )
)
job.result()

print(f"Loaded {job.output_rows} rows into {table_id}")

Shape: (296, 2)
Unique regions: ['East Midlands', 'East of England', 'London', 'North East', 'North West', 'South East', 'South West', 'West Midlands', 'Yorkshire and The Humber']
Loaded 296 rows into deprivation-spending-analysis.raw_data.raw_region_lookup
